In [45]:
from google.cloud import bigquery
import pandas as pd
from typing import Union
import re
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, DataTable, TableColumn, CustomJS, Select, Button, Div
from bokeh.plotting import show, output_file, save
from bokeh.layouts import column, row

In [3]:
output_notebook()

Loading BokehJS ...

In [4]:
client = bigquery.Client(project='subugoe-collaborative')

In [42]:
oal_inst_lower_saxony_raw = client.query(f"""
                                          SELECT DISTINCT
                                          CASE 
                                            WHEN oal.id IS NOT NULL THEN oal.id
                                            ELSE kb.id
                                          END AS id,
                                          kb.source AS kb_source, 
                                          oal.source AS oal_source,
                                          kb.inst_name AS kb_name,
                                          oal.inst_name AS oal_name,
                                          kb.ror_id,
                                          CASE 
                                            WHEN oal.publication_year IS NOT NULL THEN oal.publication_year
                                            ELSE kb.publication_year
                                          END AS publication_year,
                                          CASE 
                                            WHEN oal.raw_affiliation_string IS NOT NULL THEN oal.raw_affiliation_string
                                            ELSE address_full
                                          END AS raw_affiliation_string,
                                          ARRAY_TO_STRING(kb.current_sectors, ',') AS kb_sectors,
                                          ARRAY_TO_STRING(oal.current_sectors, ',') AS oal_sectors
                                        FROM (
                                          SELECT o.id, 
                                                 current_sectors, 
                                                 federal_states.inst_name, 
                                                 CONCAT('https://ror.org/', federal_states.ror_id) AS ror_id,
                                                 address_full, 
                                                 REGEXP_REPLACE(address_full, '[., ]', '') AS raw_affiliation_string_cleaned,
                                                 publication_year, 
                                                 'KB' AS source
                                          FROM `subugoe-collaborative.openbib.kb_a_addr_inst` AS inst
                                          JOIN `subugoe-collaborative.openbib.kb_inst_lookup` AS kb_inst
                                            ON inst.inst_id_top = kb_inst.inst_id
                                          JOIN `subugoe-collaborative.resources.inst_with_federal_state` AS federal_states
                                            ON CASE
                                                  WHEN kb_inst.inst_id = 621 THEN 'https://ror.org/03m2kj587'
                                                  ELSE kb_inst.ror 
                                                END = CONCAT('https://ror.org/', federal_states.ror_id)
                                          JOIN `subugoe-collaborative.openalex_walden.works` AS o
                                              ON inst.openalex_id = o.id
                                          WHERE publication_year BETWEEN 2020 AND 2024 
                                            AND type IN ('article', 'review')
                                            AND is_xpac=FALSE
                                            AND federal_states.federal_state = 'Niedersachsen'
                                        ) AS kb
                                        FULL OUTER JOIN (
                                        SELECT o.id, 
                                               current_sectors, 
                                               federal_states.inst_name, 
                                               raw_affiliation_string,
                                               REGEXP_REPLACE(raw_affiliation_string, '[., ]', '') AS raw_affiliation_string_cleaned,
                                               publication_year, 
                                               'OAL' AS source
                                          FROM (
                                            SELECT oal.id, 
                                                    publication_year,
                                                    CASE 
                                                    WHEN inst.ror = 'https://ror.org/021ft0n22' THEN 'https://ror.org/01y9bpm73'
                                                    ELSE inst.ror
                                                    END AS ror,
                                                    aff.raw_affiliation_string
                                            FROM `subugoe-collaborative.openalex_walden.works` AS oal
                                            LEFT JOIN UNNEST(authorships) AS aut
                                            LEFT JOIN UNNEST(aut.institutions) AS inst
                                            LEFT JOIN UNNEST(aut.affiliations) AS aff
                                            WHERE publication_year BETWEEN 2020 AND 2024 
                                            AND oal.type IN ('article', 'review')
                                            AND is_xpac=FALSE
                                          ) AS o
                                          JOIN `subugoe-collaborative.resources.inst_with_federal_state` AS federal_states
                                              ON o.ror = CONCAT('https://ror.org/', federal_states.ror_id)
                                          LEFT JOIN `subugoe-collaborative.openbib.kb_inst_lookup` AS kb_inst
                                              ON kb_inst.ror = CONCAT('https://ror.org/', federal_states.ror_id)
                                          WHERE federal_states.federal_state = 'Niedersachsen'
                                        ) AS oal
                                        ON kb.id = oal.id
                                        AND LOWER(kb.raw_affiliation_string_cleaned) = LOWER(oal.raw_affiliation_string_cleaned)
                            """).to_dataframe()

In [5]:
oal_inst_lower_saxony_raw = client.query(f"""
                                          SELECT DISTINCT
                                          CASE 
                                            WHEN oal.id IS NOT NULL THEN oal.id
                                            ELSE kb.id
                                          END AS id,
                                          kb.source AS kb_source, 
                                          oal.source AS oal_source,
                                          kb.inst_name AS kb_name,
                                          oal.inst_name AS oal_name,
                                          kb.ror_id,
                                          CASE 
                                            WHEN oal.publication_year IS NOT NULL THEN oal.publication_year
                                            ELSE kb.publication_year
                                          END AS publication_year,
                                          CASE 
                                            WHEN oal.raw_affiliation_string IS NOT NULL THEN oal.raw_affiliation_string
                                            ELSE address_full
                                          END AS raw_affiliation_string
                                        FROM (
                                          SELECT o.id, 
                                                 CASE 
                                                   WHEN kb_inst.ror = 'https://ror.org/021ft0n22' THEN 'Universitätsmedizin Göttingen'
                                                   WHEN kb_inst.ror = 'https://ror.org/02w2y2t16' THEN 'Leuphana Universität Lüneburg'
                                                   WHEN kb_inst.ror = 'https://ror.org/033n9gh91' THEN 'Carl von Ossietzky Universität Oldenburg'
                                                   WHEN kb_inst.ror = 'https://ror.org/02vvvm705' THEN 'Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth'
                                                   WHEN kb_inst.ror = 'https://ror.org/01bc76c69' THEN 'Hochschule Emden/Leer'
                                                   WHEN kb_inst.ror = 'https://ror.org/0304hq317' THEN 'Gottfried Wilhelm Leibniz Universität Hannover'
                                                   WHEN kb_inst.ror = 'https://ror.org/00x67m532' THEN 'Hochschule für Musik, Theater und Medien Hannover'
                                                   WHEN kb_inst.ror = 'https://ror.org/03m2kj587' THEN 'Hochschule Hannover'
                                                   WHEN kb_inst.ror = 'https://ror.org/015qjqf64' THEN 'Stiftung Tierärztliche Hochschule Hannover'
                                                   WHEN kb_inst.ror = 'https://ror.org/00f2yqf98' THEN 'Medizinische Hochschule Hannover (MHH)'
                                                   WHEN kb_inst.ror = 'https://ror.org/00f5q5839' THEN 'HAWK Hochschule für angewandte Wissenschaft und Kunst'
                                                   WHEN kb_inst.ror = 'https://ror.org/02f9det96' THEN 'Stiftung Universität Hildesheim'
                                                   WHEN kb_inst.ror = 'https://ror.org/01y9bpm73' THEN 'Georg-August-Universität Göttingen'
                                                   WHEN kb_inst.ror = 'https://ror.org/010nsgg66' THEN 'Technische Universität Braunschweig'
                                                   WHEN kb_inst.ror = 'https://ror.org/03aft2f80' THEN 'Hochschule für Bildende Künste Braunschweig'
                                                   WHEN kb_inst.ror = 'https://ror.org/01bk10867' THEN 'Ostfalia Hochschule für angewandte Wissenschaften'
                                                   WHEN kb_inst.ror = 'https://ror.org/04qb8nc58' THEN 'Technische Universität Clausthal'
                                                   WHEN kb_inst.ror = 'https://ror.org/04qmmjx98' THEN 'Universität Osnabrück'
                                                   WHEN kb_inst.ror = 'https://ror.org/059vymd37' THEN 'Hochschule Osnabrück'
                                                   WHEN kb_inst.ror = 'https://ror.org/045y6d111' THEN 'Universität Vechta'
                                                   -- Ergänzung von Suborganisationen (Oldenburg)
                                                   WHEN inst_id = 445 THEN 'Carl von Ossietzky Universität Oldenburg' -- Evangelisches Krankenhaus Oldenburg
                                                   WHEN inst_id = 6612 THEN 'Carl von Ossietzky Universität Oldenburg' -- UMO - Universitätsmedizin Oldenburg
                                                   WHEN inst_id = 316 THEN 'Carl von Ossietzky Universität Oldenburg' -- Klinikum Oldenburg gGmbH
                                                   WHEN inst_id = 315 THEN 'Carl von Ossietzky Universität Oldenburg' -- Pius-Hospital Oldenburg
                                                   WHEN inst_id = 5488 THEN 'Carl von Ossietzky Universität Oldenburg' -- Helmholtz-Institut für Funktionelle Marine Biodiversität an der Universität Oldenburg (HIFMB)
                                                   WHEN inst_id = 4564 THEN 'Carl von Ossietzky Universität Oldenburg' -- Institute for Science Networking Oldenburg GmbH
                                                   ELSE ''
                                                 END AS inst_name,
                                                 kb_inst.ror AS ror_id,
                                                 address_full, 
                                                 REGEXP_REPLACE(address_full, '[., ]', '') AS raw_affiliation_string_cleaned,
                                                 publication_year, 
                                                 'KB' AS source
                                          FROM `subugoe-collaborative.resources.kb_a_addr_inst_202601` AS inst
                                          JOIN `subugoe-collaborative.resources.add_institution_lookup_kb_suppl_202601` AS kb_inst
                                            ON inst.inst_id_top = kb_inst.inst_id
                                          JOIN `subugoe-collaborative.openalex_walden.works` AS o
                                              ON CONCAT('https://openalex.org/', inst.item_id) = o.id
                                          WHERE o.type IN ('article', 'review') 
                                              AND is_paratext=FALSE 
                                              AND is_retracted=FALSE 
                                              AND is_xpac=FALSE
                                              AND publication_year BETWEEN 2020 AND 2024
                                              AND (kb_inst.ror IN (
                                                  'https://ror.org/021ft0n22', -- Universitätsmedizin Göttingen
                                                  'https://ror.org/02w2y2t16', -- Leuphana Universität Lüneburg
                                                  'https://ror.org/033n9gh91', -- Carl von Ossietzky Universität Oldenburg
                                                  'https://ror.org/02vvvm705', -- Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth
                                                  'https://ror.org/01bc76c69', -- Hochschule Emden/Leer
                                                  'https://ror.org/0304hq317', -- Gottfried Wilhelm Leibniz Universität Hannover
                                                  'https://ror.org/00x67m532', -- Hochschule für Musik, Theater und Medien Hannover
                                                  'https://ror.org/03m2kj587', -- Hochschule Hannover
                                                  'https://ror.org/015qjqf64', -- Stiftung Tierärztliche Hochschule Hannover
                                                  'https://ror.org/00f2yqf98', -- Medizinische Hochschule Hannover (MHH)
                                                  'https://ror.org/00f5q5839', -- HAWK Hochschule für angewandte Wissenschaft und Kunst
                                                  'https://ror.org/02f9det96', -- Stiftung Universität Hildesheim
                                                  'https://ror.org/01y9bpm73', -- Georg-August-Universität Göttingen
                                                  'https://ror.org/010nsgg66', -- Technische Universität Braunschweig
                                                  'https://ror.org/03aft2f80', -- Hochschule für Bildende Künste Braunschweig
                                                  'https://ror.org/01bk10867', -- Ostfalia Hochschule für angewandte Wissenschaften
                                                  'https://ror.org/04qb8nc58', -- Technische Universität Clausthal
                                                  'https://ror.org/04qmmjx98', -- Universität Osnabrück
                                                  'https://ror.org/059vymd37', -- Hochschule Osnabrück
                                                  'https://ror.org/045y6d111' -- Universität Vechta
                                              ) OR kb_inst.inst_id IN (
                                                    -- Ergänzung von Suborganisationen (Oldenburg)
                                                    445, -- 'Evangelisches Krankenhaus Oldenburg'
                                                    6612, -- 'UMO - Universitätsmedizin Oldenburg'
                                                    316, -- 'Klinikum Oldenburg gGmbH'
                                                    315, -- 'Pius-Hospital Oldenburg'
                                                    5488, -- 'Helmholtz-Institut für Funktionelle Marine Biodiversität an der Universität Oldenburg (HIFMB)'
                                                    4564 -- Institute for Science Networking Oldenburg GmbH
                                                )
                                              )
                                        ) AS kb
                                        FULL OUTER JOIN (
                                            SELECT oal.id, 
                                               CASE 
                                                 WHEN inst.ror = 'https://ror.org/02w2y2t16' THEN 'Leuphana Universität Lüneburg'
                                                 WHEN inst.ror = 'https://ror.org/033n9gh91' THEN 'Carl von Ossietzky Universität Oldenburg'
                                                 WHEN inst.ror = 'https://ror.org/02vvvm705' THEN 'Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth'
                                                 WHEN inst.ror = 'https://ror.org/01bc76c69' THEN 'Hochschule Emden/Leer'
                                                 WHEN inst.ror = 'https://ror.org/0304hq317' THEN 'Gottfried Wilhelm Leibniz Universität Hannover'
                                                 WHEN inst.ror = 'https://ror.org/00x67m532' THEN 'Hochschule für Musik, Theater und Medien Hannover'
                                                 WHEN inst.ror = 'https://ror.org/03m2kj587' THEN 'Hochschule Hannover'
                                                 WHEN inst.ror = 'https://ror.org/015qjqf64' THEN 'Stiftung Tierärztliche Hochschule Hannover'
                                                 WHEN inst.ror = 'https://ror.org/00f2yqf98' THEN 'Medizinische Hochschule Hannover (MHH)'
                                                 WHEN inst.ror = 'https://ror.org/00f5q5839' THEN 'HAWK Hochschule für angewandte Wissenschaft und Kunst'
                                                 WHEN inst.ror = 'https://ror.org/02f9det96' THEN 'Stiftung Universität Hildesheim'
                                                 WHEN inst.ror = 'https://ror.org/01y9bpm73' THEN 'Georg-August-Universität Göttingen'
                                                 WHEN inst.ror = 'https://ror.org/010nsgg66' THEN 'Technische Universität Braunschweig'
                                                 WHEN inst.ror = 'https://ror.org/03aft2f80' THEN 'Hochschule für Bildende Künste Braunschweig'
                                                 WHEN inst.ror = 'https://ror.org/01bk10867' THEN 'Ostfalia Hochschule für angewandte Wissenschaften'
                                                 WHEN inst.ror = 'https://ror.org/04qb8nc58' THEN 'Technische Universität Clausthal'
                                                 WHEN inst.ror = 'https://ror.org/04qmmjx98' THEN 'Universität Osnabrück'
                                                 WHEN inst.ror = 'https://ror.org/059vymd37' THEN 'Hochschule Osnabrück'
                                                 WHEN inst.ror = 'https://ror.org/045y6d111' THEN 'Universität Vechta'
                                                 -- Ergänzung von Suborganisationen (GAU)
                                                 WHEN inst.ror = 'https://ror.org/021ft0n22' THEN 'Georg-August-Universität Göttingen' -- UMG in GAU integrieren für Vergleichbarkeit
                                                 WHEN inst.ror = 'https://ror.org/00cd95c65' THEN 'Georg-August-Universität Göttingen' -- Gesellschaft für wissenschaftliche Datenverarbeitung mbH Göttingen
                                                 WHEN inst.ror = 'https://ror.org/02f04tm31' THEN 'Georg-August-Universität Göttingen' -- Göttingen Campus Institut für Dynamic biologischer Netzwerke
                                                 WHEN inst.ror = 'https://ror.org/044sxzm68' THEN 'Georg-August-Universität Göttingen' -- Campus-Institut Data Science (CIDAS)
                                                 WHEN inst.ror = 'https://ror.org/05745n787' THEN 'Georg-August-Universität Göttingen' -- Niedersächsische Staats-und Universitätsbibliothek Göttingen
                                                 WHEN inst.ror = 'https://ror.org/03vwt8p73' THEN 'Georg-August-Universität Göttingen' -- Else Kröner Fresenius Zentrum für Optogenetische Therapien
                                                 -- Ergänzung von Suborganisationen (LUH)
                                                 WHEN inst.ror = 'https://ror.org/039t4wk02' THEN 'Gottfried Wilhelm Leibniz Universität Hannover' -- Forschungszentrum L3S
                                                 WHEN inst.ror = 'https://ror.org/00w53fs94' THEN 'Gottfried Wilhelm Leibniz Universität Hannover' -- Forschungszentrum Küste (FZK)
                                                 -- Ergänzung von Suborganisationen (Oldenburg)
                                                 WHEN inst.ror = 'https://ror.org/025t8vx68' THEN 'Carl von Ossietzky Universität Oldenburg' -- Institut für Ökonomische Bildung
                                                 WHEN inst.ror = 'https://ror.org/0060pja03' THEN 'Carl von Ossietzky Universität Oldenburg' -- Institut für Chemie und Biologie des Meeres
                                                 WHEN inst.ror = 'https://ror.org/01t0n2c80' THEN 'Carl von Ossietzky Universität Oldenburg' -- Klinikum Oldenburg
                                                 WHEN inst.ror = 'https://ror.org/04830hf15' THEN 'Carl von Ossietzky Universität Oldenburg' -- Evangelisches Krankenhaus Oldenburg
                                                 WHEN inst.ror = 'https://ror.org/03avbdx23' THEN 'Carl von Ossietzky Universität Oldenburg' -- Pius Hospital Oldenburg
                                                 WHEN inst.ror = 'https://ror.org/00tea5y39' THEN 'Carl von Ossietzky Universität Oldenburg' -- Helmholtz-Institut für Funktionelle Marine Biodiversität
                                                 ELSE ''
                                               END AS inst_name,
                                               inst.ror AS oal_id,
                                               raw_affiliation_string,
                                               REGEXP_REPLACE(raw_affiliation_string, '[., ]', '') AS raw_affiliation_string_cleaned,
                                               publication_year, 
                                               'OAL' AS source
                                            FROM `subugoe-collaborative.openalex_walden.works` AS oal
                                            LEFT JOIN UNNEST(authorships) AS aut
                                            LEFT JOIN UNNEST(aut.institutions) AS inst
                                            LEFT JOIN UNNEST(aut.affiliations) AS aff
                                            WHERE oal.type IN ('article', 'review') 
                                                AND is_paratext=FALSE 
                                                AND is_retracted=FALSE 
                                                AND is_xpac=FALSE
                                                AND publication_year BETWEEN 2020 AND 2024
                                                AND inst.ror IN (
                                                    'https://ror.org/021ft0n22', -- Universitätsmedizin Göttingen
                                                    'https://ror.org/02w2y2t16', -- Leuphana Universität Lüneburg
                                                    'https://ror.org/033n9gh91', -- Carl von Ossietzky Universität Oldenburg
                                                    'https://ror.org/02vvvm705', -- Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth
                                                    'https://ror.org/01bc76c69', -- Hochschule Emden/Leer
                                                    'https://ror.org/0304hq317', -- Gottfried Wilhelm Leibniz Universität Hannover
                                                    'https://ror.org/00x67m532', -- Hochschule für Musik, Theater und Medien Hannover
                                                    'https://ror.org/03m2kj587', -- Hochschule Hannover
                                                    'https://ror.org/015qjqf64', -- Stiftung Tierärztliche Hochschule Hannover
                                                    'https://ror.org/00f2yqf98', -- Medizinische Hochschule Hannover (MHH)
                                                    'https://ror.org/00f5q5839', -- HAWK Hochschule für angewandte Wissenschaft und Kunst
                                                    'https://ror.org/02f9det96', -- Stiftung Universität Hildesheim
                                                    'https://ror.org/01y9bpm73', -- Georg-August-Universität Göttingen
                                                    'https://ror.org/010nsgg66', -- Technische Universität Braunschweig
                                                    'https://ror.org/03aft2f80', -- Hochschule für Bildende Künste Braunschweig
                                                    'https://ror.org/01bk10867', -- Ostfalia Hochschule für angewandte Wissenschaften
                                                    'https://ror.org/04qb8nc58', -- Technische Universität Clausthal
                                                    'https://ror.org/04qmmjx98', -- Universität Osnabrück
                                                    'https://ror.org/059vymd37', -- Hochschule Osnabrück
                                                    'https://ror.org/045y6d111', -- Universität Vechta
                                                    -- Ergänzung von Suborganisationen (GAU)
                                                    'https://ror.org/00cd95c65', -- Gesellschaft für wissenschaftliche Datenverarbeitung mbH Göttingen
                                                    'https://ror.org/02f04tm31', -- Göttingen Campus Institut für Dynamic biologischer Netzwerke
                                                    'https://ror.org/044sxzm68', -- Campus-Institut Data Science (CIDAS)
                                                    'https://ror.org/05745n787', -- Niedersächsische Staats-und Universitätsbibliothek Göttingen
                                                    'https://ror.org/03vwt8p73', -- Else Kröner Fresenius Zentrum für Optogenetische Therapien
                                                    -- Ergänzung von Suborganisationen (LUH)
                                                    'https://ror.org/039t4wk02', -- Forschungszentrum L3S
                                                    'https://ror.org/00w53fs94', -- Forschungszentrum Küste (FZK)
                                                    -- Ergänzung von Suborganisationen (Oldenburg)
                                                    'https://ror.org/025t8vx68', -- Institut für Ökonomische Bildung
                                                    'https://ror.org/0060pja03', -- Institut für Chemie und Biologie des Meeres
                                                    'https://ror.org/01t0n2c80', -- Klinikum Oldenburg
                                                    'https://ror.org/04830hf15' -- Evangelisches Krankenhaus Oldenburg
                                                    'https://ror.org/03avbdx23' -- Pius Hospital Oldenburg
                                                    'https://ror.org/00tea5y39' -- Helmholtz-Institut für Funktionelle Marine Biodiversität
                                                )
                                            ) AS oal
                                        ON kb.id = oal.id
                                        AND LOWER(kb.raw_affiliation_string_cleaned) = LOWER(oal.raw_affiliation_string_cleaned)
                            """).to_dataframe()

In [7]:
#oal_inst_lower_saxony_raw.to_csv('../data/inst_list_full_with_aff_strings_cleaned.csv', sep=',', index=False)

In [170]:
oal_inst_lower_saxony_raw = pd.read_csv('../data/inst_list_full_with_aff_strings_cleaned.csv')

In [171]:
oal_inst_lower_saxony = oal_inst_lower_saxony_raw.copy()

In [172]:
oal_inst_lower_saxony.head()

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
0,https://openalex.org/W4405598901,KB,OAL,"Hochschule für Musik, Theater und Medien Hannover","Hochschule für Musik, Theater und Medien Hannover",https://ror.org/00x67m532,2024,"Hochschule für Musik, Theater und Medien Hanno..."
1,https://openalex.org/W4213176444,KB,OAL,Universität Vechta,Universität Vechta,https://ror.org/045y6d111,2022,"Faculty of Psychology, University of Vechta, V..."
2,https://openalex.org/W4403043964,KB,OAL,"Hochschule für Musik, Theater und Medien Hannover","Hochschule für Musik, Theater und Medien Hannover",https://ror.org/00x67m532,2024,"Hanover Music Lab, Hochschule für Musik, Theat..."
3,https://openalex.org/W3113488469,KB,OAL,Ostfalia Hochschule für angewandte Wissenschaften,Ostfalia Hochschule für angewandte Wissenschaften,https://ror.org/01bk10867,2020,"Faculty of Computer Science / IT, Salzdahlumer..."
4,https://openalex.org/W4307201653,KB,OAL,Hochschule Hannover,Gottfried Wilhelm Leibniz Universität Hannover,https://ror.org/03m2kj587,2022,"Research Group Pharmaceutical Biotechnology, F..."


In [173]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W3045999247'].raw_affiliation_string.tolist()

[]

In [174]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W3012291016'].raw_affiliation_string.tolist()

['Klinik für Strahlentherapie und Spezielle Onkologie, Medizinische Hochschule Hannover, Hannover, Deutschland']

In [175]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W3012291016']

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
143030,https://openalex.org/W3012291016,KB,OAL,Medizinische Hochschule Hannover (MHH),Medizinische Hochschule Hannover (MHH),https://ror.org/00f2yqf98,2020,Klinik für Strahlentherapie und Spezielle Onko...


In [176]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W3005853386']

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
120590,https://openalex.org/W3005853386,KB,OAL,Leuphana Universität Lüneburg,Leuphana Universität Lüneburg,https://ror.org/02w2y2t16,2020,Leuphana University of Lüneburg


In [177]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W3005853386'].raw_affiliation_string.tolist()

['Leuphana University of Lüneburg']

In [178]:
'Leuphana University of Lüneburg' == ' Leuphana University of Lüneburg'

False

In [179]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W4292018798']

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string


In [180]:
kb_list = oal_inst_lower_saxony.groupby(['id', 'raw_affiliation_string', 'publication_year'])['kb_name'].apply(set).reset_index()

In [181]:
oal_list = oal_inst_lower_saxony.groupby(['id', 'raw_affiliation_string', 'publication_year'])['oal_name'].apply(set).reset_index()

In [182]:
inst_list = pd.merge(kb_list, oal_list, on=['id', 'raw_affiliation_string', 'publication_year'], how='outer')

In [183]:
inst_list['kb_name'] = inst_list['kb_name'].fillna('').apply(set)
inst_list['oal_name'] = inst_list['oal_name'].fillna('').apply(set)

inst_list['in_oal_missing'] = list(inst_list['kb_name'] - inst_list['oal_name'])
inst_list['in_kb_missing'] = list(inst_list['oal_name'] - inst_list['kb_name'])

inst_list['kb_count'] = inst_list.kb_name.str.len()
inst_list['oal_count'] = inst_list.oal_name.str.len()

In [184]:
inst_list.head()

,id,raw_affiliation_string,publication_year,kb_name,oal_name,in_oal_missing,in_kb_missing,kb_count,oal_count
0,https://openalex.org/W107125650,"Institute for Environmental Communication, Leu...",2020,{Leuphana Universität Lüneburg},{Leuphana Universität Lüneburg},{},{},1,1
1,https://openalex.org/W112007689,Götting KG,2024,{nan},{Georg-August-Universität Göttingen},{nan},{Georg-August-Universität Göttingen},1,1
2,https://openalex.org/W112007689,INSTITUT FÜR TRANSPORT-UND AUTOMATISIERUNGSTEC...,2024,{nan},"{Georg-August-Universität Göttingen, Gottfried...",{nan},"{Georg-August-Universität Göttingen, Gottfried...",1,2
3,https://openalex.org/W112007689,Institut für Transport- und Automatisierungste...,2024,{Gottfried Wilhelm Leibniz Universität Hannover},{Gottfried Wilhelm Leibniz Universität Hannover},{},{},1,1
4,https://openalex.org/W1483587807,"Centre Georg Simmel, Recherches franco-alleman...",2021,{nan},{Leuphana Universität Lüneburg},{nan},{Leuphana Universität Lüneburg},1,1


In [185]:
inst_list[['id', 'raw_affiliation_string', 'publication_year', 'in_oal_missing', 'in_kb_missing']].to_csv('../data/inst_list_full.csv', sep=',', index=False)

In [186]:
df = pd.read_csv('../data/inst_list_full.csv')

In [187]:
df['in_oal_missing'] = df['in_oal_missing'].replace('set()', None)
df['in_oal_missing'] = df['in_oal_missing'].replace('{None}', None)
df['in_oal_missing'] = df['in_oal_missing'].replace('{None}', None)
df['in_oal_missing'] = df['in_oal_missing'].str.replace('{', '')
df['in_oal_missing'] = df['in_oal_missing'].str.replace('}', '')
df['in_oal_missing'] = df['in_oal_missing'].str.replace("'", '')

df['in_kb_missing'] = df['in_kb_missing'].replace('set()', None)
df['in_kb_missing'] = df['in_kb_missing'].replace('{None}', None)
df['in_kb_missing'] = df['in_kb_missing'].replace('{None}', None)
df['in_kb_missing'] = df['in_kb_missing'].str.replace('{', '')
df['in_kb_missing'] = df['in_kb_missing'].str.replace('}', '')
df['in_kb_missing'] = df['in_kb_missing'].str.replace("'", '')

In [188]:
df = df.assign(in_oal_missing=df['in_oal_missing'].str.split(',')).explode('in_oal_missing').reset_index(drop=True)

In [166]:
filter_uni_fh = client.query(f"""
                                SELECT DISTINCT inst_name, dfg_inst_id, ror_id
                                FROM `subugoe-collaborative.resources.inst_with_federal_state` AS f
                                JOIN `subugoe-collaborative.openbib.kb_inst_lookup` AS kb
                                    ON CASE
                                        WHEN kb.inst_id = 621 THEN 'https://ror.org/03m2kj587'
                                        ELSE kb.ror 
                                    END = CONCAT('https://ror.org/', f.ror_id)
                                LEFT JOIN UNNEST(current_sectors) AS sector
                                WHERE federal_state = 'Niedersachsen' 
                                  AND sector IN ('uni', 'fh', 'khmh') 
                                  AND dfg_inst_id NOT IN (
                                    220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                                    13033, -- PFH Private Hochschule Göttingen
                                    233118106, -- Leibniz-Fachhochschule
                                    198800578, -- Hochschule 21 Buxtehude
                                    195374963 -- Hochschule Weserbergland
                                  )
                              """).to_dataframe()

In [167]:
filter_uni_fh.replace(
    {'Hochschule für angewandte Wissenschaft und Kunst Hildesheim/Holzminden/Göttingen': 'HAWK Hochschule für angewandte Wissenschaft und Kunst'},
    inplace=True)
filter_uni_fh.replace(
    {'https://ror.org/03z6vda50': 'https://ror.org/03m2kj587'},
    inplace=True)

In [169]:
#filter_uni_fh.to_csv('../data/filter_uni_fh.csv', sep=',', index=False)

In [189]:
filter_uni_fh = pd.read_csv('../data/filter_uni_fh.csv')

In [190]:
df = pd.merge(df, filter_uni_fh, how='right', left_on='in_oal_missing', right_on='inst_name')

In [191]:
df = df[~df.publication_year.isnull()]
df['publication_year'] = df['publication_year'].astype(int)

In [192]:
df.head()

,id,raw_affiliation_string,publication_year,in_oal_missing,in_kb_missing,inst_name,dfg_inst_id,ror_id
0,https://openalex.org/W173619620,Universität Lüneburg Zentrum für Angewandte Ge...,2024,Leuphana Universität Lüneburg,nan,Leuphana Universität Lüneburg,10232,02w2y2t16
1,https://openalex.org/W2982542118,German Institute for Economic Research (DIW Be...,2022,Leuphana Universität Lüneburg,nan,Leuphana Universität Lüneburg,10232,02w2y2t16
2,https://openalex.org/W3111444816,"Institute of Psychology, Leuphana University L...",2020,Leuphana Universität Lüneburg,nan,Leuphana Universität Lüneburg,10232,02w2y2t16
3,https://openalex.org/W3137323162,Institute for Sustainable Development and Lear...,2021,Leuphana Universität Lüneburg,nan,Leuphana Universität Lüneburg,10232,02w2y2t16
4,https://openalex.org/W3175971255,"Center for the Study of Democracy, Leuphana Un...",2021,Leuphana Universität Lüneburg,nan,Leuphana Universität Lüneburg,10232,02w2y2t16


In [193]:
true_df = df[~df.in_oal_missing.isnull()].copy()
true_df['in_kb_missing'] = true_df['in_kb_missing'].replace('nan', '')

In [194]:
true_df.id.count()

6293

In [195]:
# Universitätsmedizin Göttingen
# https://ror.org/021ft0n22 
umg = ['univ(ersi(t)?((ä|a|ae)ts|y))?(\s|\sof\s)?(med|hosp|klinik)', 'gynäkologie',
       'medical (university )?(cent(re|er)|school|clinic|science)?', 'neuroimmunology', 'cardiology', 
       'h((a)?e|ä)matolog(ie|y)', 'radiolog(ie|y)', 'neurology', 'medical (bio)?informatics', 'pneumology', 'augenklinik',
       'pharmacology', 'an(a)?esthesiology', 'bioimaging', 'psychiatry', 'psychoneuroscience', 'dental', 'anatomy',
       'heart cent(er|re)', 'palliativmedizin', 'cellular biophysics', 'medizinische statistik', 'nephrology',
       'infektiologie', '(trauma|neuro|thoracic)(\s)?surgery', 'epidemiology', 'medizin und psychotherapie',
       'otorhinolaryngology', 'maxillofacial', 'jugendpsychiatrie', 'molecular cell', 'gastroenterology',
       'otolaryngology', '889', '1690', '1286', 'faculty of medicine', 'department of dermatology']

In [196]:
def map_umg(address: str) -> Union[str, None]:
    pattern = re.compile(r'|'.join(umg), re.IGNORECASE)
    res = bool(pattern.search(address))
    if res:
        return  'Universitätsmedizin Göttingen'
    else:
        return 'Georg-August-Universität Göttingen'

In [197]:
true_df.loc[true_df['in_oal_missing'] == 'Georg-August-Universität Göttingen', 'in_oal_missing'] = \
           true_df.loc[true_df['in_oal_missing'] == 'Georg-August-Universität Göttingen']['raw_affiliation_string'].apply(map_umg)

In [198]:
source = ColumnDataSource(data=true_df)

In [199]:
output_file(filename='download.html', title='HDN-FIS: Institutionen Download')

columns = [
    TableColumn(field='id', title='OpenAlex ID', width=200),
    TableColumn(field='publication_year', title='Publikationsjahr', width=100),
    TableColumn(field='raw_affiliation_string', width=700, title='Affiliationsstring'),
    TableColumn(field='in_oal_missing', title='Fehlende Institution in OpenAlex'),
    TableColumn(field='in_kb_missing', title='Institutionszuordnung durch OpenAlex')
]

data_table = DataTable(source=source, 
                       columns=columns, 
                       editable=True, 
                       selectable=True,
                       row_height=35,
                       width=1600,
                       height=500
                      )

source.data = dict(true_df[true_df.in_oal_missing == 'Carl von Ossietzky Universität Oldenburg'])

select_inst = Select(title='Institution', 
                     width=400,
                     value='Carl von Ossietzky Universität Oldenburg', 
                     options=list(true_df.in_oal_missing.unique()))

select_year = Select(title='Publikationsjahr', 
                     width=200,
                     value='Alle', 
                     options=['Alle'] + [str(year) for year in sorted(list(true_df.publication_year.unique()))])

div_row_count = Div(text=f"Treffer: <b>{len(source.data['in_oal_missing'])}</b>")

callback = CustomJS(args=dict(source=source, 
                              row_div=div_row_count,
                              year=select_year, 
                              inst=select_inst, 
                              original_data=true_df.to_dict('list')), 
                    code="""
    let filtered_data = {
        'index': [],
        'id': [],
        'publication_year': [],
        'raw_affiliation_string': [],
        'in_oal_missing': [],
        'in_kb_missing': []
    };

    if (year.value == 'Alle') {
        for (let i = 0; i < original_data['in_oal_missing'].length; i++) {
            if (original_data['in_oal_missing'][i] == inst.value) {
                filtered_data['index'].push(original_data[i]);
                filtered_data['id'].push(original_data['id'][i]);
                filtered_data['publication_year'].push(original_data['publication_year'][i]);
                filtered_data['raw_affiliation_string'].push(original_data['raw_affiliation_string'][i]);
                filtered_data['in_oal_missing'].push(original_data['in_oal_missing'][i]);
                filtered_data['in_kb_missing'].push(original_data['in_kb_missing'][i]);
            }
        }
    } else {

        for (let i = 0; i < original_data['in_oal_missing'].length; i++) {
            if (original_data['publication_year'][i] == parseInt(year.value) && original_data['in_oal_missing'][i] == inst.value) {
                filtered_data['index'].push(original_data[i]);
                filtered_data['id'].push(original_data['id'][i]);
                filtered_data['publication_year'].push(original_data['publication_year'][i]);
                filtered_data['raw_affiliation_string'].push(original_data['raw_affiliation_string'][i]);
                filtered_data['in_oal_missing'].push(original_data['in_oal_missing'][i]);
                filtered_data['in_kb_missing'].push(original_data['in_kb_missing'][i]);
            }
        }
    }

    const row_count = filtered_data['id'].length;

    source.data = filtered_data;
    row_div.text = `Treffer: <b>${row_count}</b>`;
""")

callback_download = CustomJS(args=dict(source=source),
                             code="""
        // FROM: https://github.com/bokeh/bokeh/blob/main/examples/server/app/export_csv/download.js
        let csv = 'OpenAlex ID;Publikationsjahr;Affiliationsstring;Fehlende Institutionen in OpenAlex\\n';
        for (let i = 0; i < source.data['id'].length; i++) {
            csv += source.data['id'][i] + ';' + source.data['publication_year'][i] + ';' + source.data['raw_affiliation_string'][i] + ';' + source.data['in_oal_missing'][i] + '\\n';
        }     

        const filename = 'FehlendeInstitutionen.csv'
        const blob = new Blob([csv], {type: 'text/csv;charset=utf-8;'})
        
        //addresses IE
        if (navigator.msSaveBlob) {
            navigator.msSaveBlob(blob, filename)
        } else {
            const link = document.createElement('a')
            link.href = URL.createObjectURL(blob)
            link.download = filename
            link.target = '_blank'
            link.style.visibility = 'hidden'
            link.dispatchEvent(new MouseEvent('click'))
        }
""")

select_inst.js_on_change('value', callback)
select_year.js_on_change('value', callback)

button = Button(label='Download', button_type='default', margin=(22, 0, 0, 15))
button.js_on_event('button_click', callback_download)

div_header = Div(text="""
                      <h1 style=font-size:16px>HDN-FIS: Institutionsanreicherung niedersächsicher Hochschulen in OpenAlex - Download</h1>
                      """
                )

div_text = Div(text="""
                    <p>Die folgende Tabelle enthält Publikationen, bei denen die Institutionszuordnung im KB unterschiedlich zu der in OpenAlex ist. <br>
                    Es werden nur Publikationen angezeigt, die eine Zuordnung zu einer niedersächsischen Hochschule haben und zwischen 2020 und 2024 erschienen sind. </p>
                    <br>
                    <p><b>Datenquellen:</b></p>
                    <ul>
                        <li><b>OpenAlex:</b> Stand März 2026</li>
                        <li><b>KB:</b> Stand Januar 2026</li>
                    </ul>
                    """
                )

#layout = column(row(select_inst, select_year, button, spacing=50), data_table, spacing=15)
layout = column(
            column(div_header, div_text), 
            row(select_inst, select_year, button, spacing=50), 
            div_row_count, 
            data_table, 
            spacing=15
)

show(layout)
save(layout)

'/Users/naustica/Desktop/lower_saxony_institutions/notebooks/download.html'

In [200]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W3045881206'].raw_affiliation_string.tolist()

['Universität Hildesheim']

In [201]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W3045881206']

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
152937,https://openalex.org/W3045881206,KB,OAL,Stiftung Universität Hildesheim,Stiftung Universität Hildesheim,https://ror.org/02f9det96,2020,Universität Hildesheim


In [202]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W3006949113'].raw_affiliation_string.tolist()

['Department of Zoology, University of Cambridge, Cambridge, UK',
 'Institute of Entomology, Biology Centre of the Czech Academy of Sciences, České Budějovice, Czech Republic',
 'Department of Evolutionary Developmental Genetics, University of Göttingen, Göttingen, Germany',
 '4Department of Developmental Biology, University of Göttingen, Göttingen, Germany']

In [203]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W3006949113']

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
16930,https://openalex.org/W3006949113,NaN,OAL,NaN,Georg-August-Universität Göttingen,NaN,2020,"Department of Zoology, University of Cambridge..."
36561,https://openalex.org/W3006949113,NaN,OAL,NaN,Georg-August-Universität Göttingen,NaN,2020,"Institute of Entomology, Biology Centre of the..."
77406,https://openalex.org/W3006949113,KB,OAL,Georg-August-Universität Göttingen,Georg-August-Universität Göttingen,https://ror.org/01y9bpm73,2020,Department of Evolutionary Developmental Genet...
96221,https://openalex.org/W3006949113,KB,OAL,Georg-August-Universität Göttingen,Georg-August-Universität Göttingen,https://ror.org/01y9bpm73,2020,"4Department of Developmental Biology, Universi..."


In [204]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W4378714479']

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
49170,https://openalex.org/W4378714479,NaN,OAL,NaN,Leuphana Universität Lüneburg,NaN,2023,Institute of Political Science Leuphana Univer...
120991,https://openalex.org/W4378714479,KB,OAL,Leuphana Universität Lüneburg,Leuphana Universität Lüneburg,https://ror.org/02w2y2t16,2023,"Institute of Political Science, Leuphana Unive..."


In [205]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W3002602652'].raw_affiliation_string.tolist()

['Department of Physics, Osnabrueck University, 49069 Osnabrueck, Germany']